In [ ]:
!pip install -q scikit-posthocs


In [ ]:
import os
import glob
import itertools
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from scipy import ndimage
from scipy.stats import linregress, shapiro, levene, f_oneway, kruskal
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import scikit_posthocs as sp

sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)

print("Libraries loaded.")


In [ ]:
ROOT_DIR = "/content/drive/MyDrive/fractal_analysis"

MODEL_FOLDERS = {
    "Adobe_Firefly_5": "Adobe Firefly 5",
    "Flux2_Pro": "Flux.2 Pro",
    "Nano_Banana_2": "Nano Banana 2",
    "GPT_Image_2": "GPT Image 2",
}

IMG_SIZE = 512
BOX_SIZES = [2, 4, 8, 16, 32, 64, 128]
R2_THRESHOLD = 0.95
NUM_GRAY_LEVELS = 256

OUTPUT_DIR = "/content/fractal_analysis_outputs"
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

MODEL_ORDER = list(MODEL_FOLDERS.values())
PALETTE = sns.color_palette("Set2", len(MODEL_ORDER))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import re

def extract_category(filename):
    m = re.search(r'(?:^|[_\-])([ABC])(?:[_\-]|$)', filename)
    return m.group(1) if m else "Unknown"

records = []
for folder_name, display_name in MODEL_FOLDERS.items():
    folder_path = os.path.join(ROOT_DIR, folder_name)
    if not os.path.isdir(folder_path):
        print(f"[WARNING] Folder not found: {folder_path}")
        continue
    files = sorted(
        glob.glob(os.path.join(folder_path, "*.png")) +
        glob.glob(os.path.join(folder_path, "*.jpg")) +
        glob.glob(os.path.join(folder_path, "*.jpeg"))
    )
    for fp in files:
        records.append({
            "filepath": fp,
            "filename": os.path.basename(fp),
            "model": display_name,
            "category": extract_category(os.path.basename(fp)),
        })

manifest = pd.DataFrame(records)
print(f"Total images found: {len(manifest)}")
print(manifest.groupby("model").size())
assert len(manifest) > 0, "No images found -- check ROOT_DIR / MODEL_FOLDERS settings."
manifest.head()


In [ ]:
def preprocess_image(filepath, size=IMG_SIZE):
    img = Image.open(filepath).convert("RGB")
    img = ImageOps.exif_transpose(img)
    w, h = img.size
    side = min(w, h)
    left, top = (w - side) // 2, (h - side) // 2
    img = img.crop((left, top, left + side, top + side))
    img = img.resize((size, size), Image.LANCZOS)
    gray = np.array(img.convert("L"), dtype=np.uint8)
    return gray

preprocessed = {}
for _, row in manifest.iterrows():
    try:
        preprocessed[row["filepath"]] = preprocess_image(row["filepath"])
    except Exception as e:
        print(f"[ERROR] {row['filename']}: {e}")

print(f"{len(preprocessed)} / {len(manifest)} images successfully preprocessed.")


In [ ]:
fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(4 * len(MODEL_ORDER), 4))
for ax, model_name in zip(axes, MODEL_ORDER):
    sample_fp = manifest[manifest["model"] == model_name]["filepath"].iloc[0]
    ax.imshow(preprocessed[sample_fp], cmap="gray")
    ax.set_title(model_name, fontsize=11)
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "00_preprocessing_examples.png"), dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
def dbc_count(img_gray, box_size, num_gray_levels=NUM_GRAY_LEVELS):
    M = img_gray.shape[0]
    s = box_size
    s_prime = max(1, round(s * num_gray_levels / M))
    n_blocks = M // s
    trimmed = img_gray[: n_blocks * s, : n_blocks * s].astype(np.int64)
    blocks = trimmed.reshape(n_blocks, s, n_blocks, s)
    block_max = blocks.max(axis=(1, 3))
    block_min = blocks.min(axis=(1, 3))
    n_r_ij = np.floor(block_max / s_prime) - np.floor(block_min / s_prime) + 1
    return float(n_r_ij.sum())


def fractal_dimension(img_gray, box_sizes=BOX_SIZES):
    N_r = np.array([dbc_count(img_gray, s) for s in box_sizes])
    log_inv_r = np.log(1.0 / np.array(box_sizes, dtype=float))
    log_N_r = np.log(N_r)
    slope, intercept, r_value, p_value, std_err = linregress(log_inv_r, log_N_r)
    return {
        "D": slope,
        "R2": r_value ** 2,
        "intercept": intercept,
        "log_inv_r": log_inv_r,
        "log_N_r": log_N_r,
        "N_r": N_r,
    }


fractal_results = {}
for fp, img in preprocessed.items():
    fractal_results[fp] = fractal_dimension(img)

print(f"Fractal dimension computed for {len(fractal_results)} images.")


In [ ]:
def gliding_box_lacunarity(img_gray, box_size):
    img_f = img_gray.astype(np.float64)
    box_sum = ndimage.uniform_filter(img_f, size=box_size, mode="reflect") * (box_size ** 2)
    mu = box_sum.mean()
    sigma2 = box_sum.var()
    if mu == 0:
        return np.nan
    return float((sigma2 / (mu ** 2)) + 1)


lacunarity_results = {}
for fp, img in preprocessed.items():
    lac_curve = {s: gliding_box_lacunarity(img, s) for s in BOX_SIZES}
    lacunarity_results[fp] = lac_curve

print(f"Lacunarity curve computed for {len(lacunarity_results)} images.")


In [ ]:
def shadow_softness_index(img_gray):
    img_f = img_gray.astype(np.float64)
    gx = ndimage.sobel(img_f, axis=0)
    gy = ndimage.sobel(img_f, axis=1)
    grad_mag = np.hypot(gx, gy)
    edge_sharpness = grad_mag.mean()
    ssi = 1.0 / (edge_sharpness + 1e-3)
    return ssi, edge_sharpness


ssi_results = {}
for fp, img in preprocessed.items():
    ssi, sharp = shadow_softness_index(img)
    ssi_results[fp] = {"SSI": ssi, "edge_sharpness": sharp}

print(f"Shadow Softness Index computed for {len(ssi_results)} images.")


In [ ]:
rows = []
for _, m in manifest.iterrows():
    fp = m["filepath"]
    if fp not in fractal_results:
        continue
    fr = fractal_results[fp]
    lac = lacunarity_results[fp]
    ssi = ssi_results[fp]
    row = {
        "filename": m["filename"],
        "model": m["model"],
        "category": m["category"],
        "D": fr["D"],
        "R2": fr["R2"],
        "valid_fit": fr["R2"] >= R2_THRESHOLD,
        "SSI": ssi["SSI"],
        "edge_sharpness": ssi["edge_sharpness"],
    }
    for s in BOX_SIZES:
        row[f"Lac_r{s}"] = lac[s]
    row["Lac_mean"] = float(np.mean(list(lac.values())))
    rows.append(row)

df = pd.DataFrame(rows)
df["model"] = pd.Categorical(df["model"], categories=MODEL_ORDER, ordered=True)

csv_path = os.path.join(OUTPUT_DIR, "all_results.csv")
df.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")
df.head()


In [ ]:
validity_table = (
    df.groupby("model", observed=True)
    .agg(
        n_total=("valid_fit", "size"),
        n_valid=("valid_fit", "sum"),
        mean_R2=("R2", "mean"),
        min_R2=("R2", "min"),
    )
    .assign(validity_ratio=lambda d: (d["n_valid"] / d["n_total"] * 100).round(1))
)
validity_table.to_csv(os.path.join(OUTPUT_DIR, "validity_table.csv"))
print(validity_table)

invalid = df[~df["valid_fit"]]
if len(invalid) > 0:
    print(f"\n[WARNING] {len(invalid)} images have R2 < {R2_THRESHOLD}; review is recommended:")
    print(invalid[["filename", "model", "R2"]].to_string(index=False))
else:
    print(f"\nAll images meet the R2 >= {R2_THRESHOLD} validity criterion.")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df, x="model", y="R2", hue="model", order=MODEL_ORDER, hue_order=MODEL_ORDER, palette=PALETTE, legend=False, ax=ax)
sns.stripplot(data=df, x="model", y="R2", order=MODEL_ORDER, color="black", alpha=0.4, size=3, ax=ax)
ax.axhline(R2_THRESHOLD, color="red", linestyle="--", linewidth=1, label=f"Validity threshold (R2={R2_THRESHOLD})")
ax.set_xlabel("Model")
ax.set_ylabel("R2 (log-log linearity)")
ax.set_title("DBC Log-Log Regression Validity (R2) -- By Model")
ax.legend()
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "01_R2_validity_boxplot.png"), dpi=300, bbox_inches="tight")
plt.savefig(os.path.join(FIG_DIR, "01_R2_validity_boxplot.pdf"), bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, model_name in zip(axes.flat, MODEL_ORDER):
    sub = df[df["model"] == model_name].sort_values("D")
    median_row = sub.iloc[len(sub) // 2]
    fp = manifest[(manifest["model"] == model_name) & (manifest["filename"] == median_row["filename"])]["filepath"].iloc[0]
    fr = fractal_results[fp]
    ax.scatter(fr["log_inv_r"], fr["log_N_r"], color="steelblue", zorder=3)
    fit_line = fr["intercept"] + fr["D"] * fr["log_inv_r"]
    ax.plot(fr["log_inv_r"], fit_line, color="firebrick", linewidth=1.5)
    ax.set_title(f"{model_name}\nD={fr['D']:.3f}, R2={fr['R2']:.4f}", fontsize=10)
    ax.set_xlabel("log(1/r)")
    ax.set_ylabel("log(N_r)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "02_example_loglog_regression.png"), dpi=300, bbox_inches="tight")
plt.savefig(os.path.join(FIG_DIR, "02_example_loglog_regression.pdf"), bbox_inches="tight")
plt.show()


In [ ]:
desc = df.groupby("model", observed=True)[["D", "Lac_mean", "SSI"]].agg(["mean", "std", "median"]).round(4)
desc.to_csv(os.path.join(OUTPUT_DIR, "descriptive_statistics.csv"))
desc


In [ ]:
def run_group_comparison(data, dv, group_col="model", alpha=0.05):
    groups_dict = {g: sub[dv].dropna().values for g, sub in data.groupby(group_col, observed=True)}
    group_names = list(groups_dict.keys())
    group_values = list(groups_dict.values())

    print(f"\n{'='*60}\nVARIABLE: {dv}\n{'='*60}")

    normality = {}
    all_normal = True
    for g, vals in groups_dict.items():
        stat, p = shapiro(vals)
        normality[g] = p
        flag = "Normal" if p > alpha else "Not Normal"
        print(f"  Shapiro-Wilk [{g}]: p={p:.4f} -> {flag}")
        if p <= alpha:
            all_normal = False

    lev_stat, lev_p = levene(*group_values)
    homogeneous = lev_p > alpha
    print(f"  Levene (variance homogeneity): p={lev_p:.4f} -> {'Homogeneous' if homogeneous else 'Not Homogeneous'}")

    result = {"dv": dv, "normality": normality, "levene_p": lev_p}

    if all_normal and homogeneous:
        F, p = f_oneway(*group_values)
        all_vals = np.concatenate(group_values)
        grand_mean = all_vals.mean()
        ss_total = np.sum((all_vals - grand_mean) ** 2)
        ss_between = sum(len(v) * (v.mean() - grand_mean) ** 2 for v in group_values)
        eta2 = ss_between / ss_total
        print(f"\n  >> One-way ANOVA: F={F:.3f}, p={p:.4g}, eta2={eta2:.4f}")
        result.update({"test": "One-way ANOVA", "stat": F, "p": p,
                        "effect_size_name": "eta-squared", "effect_size": eta2})

        endog = np.concatenate(group_values)
        groups_labels = np.concatenate([[g] * len(v) for g, v in zip(group_names, group_values)])
        tukey = pairwise_tukeyhsd(endog, groups_labels, alpha=alpha)
        tukey_df = pd.DataFrame(data=tukey._results_table.data[1:], columns=tukey._results_table.data[0])
        print("\n  Tukey HSD post-hoc:")
        print(tukey_df.to_string(index=False))

        mat = pd.DataFrame(np.nan, index=group_names, columns=group_names)
        for _, r in tukey_df.iterrows():
            mat.loc[r["group1"], r["group2"]] = r["p-adj"]
            mat.loc[r["group2"], r["group1"]] = r["p-adj"]
        for g in group_names:
            mat.loc[g, g] = 1.0
        result["posthoc_matrix"] = mat
        result["posthoc_method"] = "Tukey HSD"

    else:
        H, p = kruskal(*group_values)
        n_total = sum(len(v) for v in group_values)
        k = len(group_values)
        eps2 = (H - k + 1) / (n_total - k)
        print(f"\n  >> Kruskal-Wallis: H={H:.3f}, p={p:.4g}, epsilon2={eps2:.4f}")
        result.update({"test": "Kruskal-Wallis", "stat": H, "p": p,
                        "effect_size_name": "epsilon-squared", "effect_size": eps2})

        long_df = data[[group_col, dv]].rename(columns={group_col: "group", dv: "value"}).dropna()
        dunn = sp.posthoc_dunn(long_df, val_col="value", group_col="group", p_adjust="holm")
        print("\n  Dunn post-hoc (Holm-corrected):")
        print(dunn.round(4).to_string())
        result["posthoc_matrix"] = dunn
        result["posthoc_method"] = "Dunn (Holm)"

    sig = "***significant difference***" if result["p"] < alpha else "no significant difference"
    print(f"\n  RESULT: {result['test']} p={result['p']:.4g} -> {sig} (alpha={alpha})")
    return result


all_results = {}
for metric in ["D", "Lac_mean", "SSI"]:
    all_results[metric] = run_group_comparison(df, metric)


In [ ]:
summary_rows = []
for metric, r in all_results.items():
    summary_rows.append({
        "Metric": metric,
        "Test": r["test"],
        "Statistic": round(r["stat"], 3),
        "p-value": f"{r['p']:.4g}",
        "Effect Size": f"{r['effect_size_name']}={r['effect_size']:.4f}",
        "Post-hoc Method": r["posthoc_method"],
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(OUTPUT_DIR, "inferential_statistics_summary.csv"), index=False)
summary_df


In [ ]:
def boxplot_metric(metric, ylabel, filename_prefix):
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.boxplot(data=df, x="model", y=metric, hue="model", order=MODEL_ORDER, hue_order=MODEL_ORDER, palette=PALETTE, legend=False, ax=ax)
    sns.stripplot(data=df, x="model", y=metric, order=MODEL_ORDER, color="black", alpha=0.4, size=3, ax=ax)
    ax.set_xlabel("Model")
    ax.set_ylabel(ylabel)
    ax.set_title(f"{ylabel} -- Distribution by Model")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"{filename_prefix}.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(FIG_DIR, f"{filename_prefix}.pdf"), bbox_inches="tight")
    plt.show()

boxplot_metric("D", "Fractal Dimension (D)", "03_fractal_dimension_boxplot")
boxplot_metric("Lac_mean", "Mean Lacunarity", "04_lacunarity_boxplot")
boxplot_metric("SSI", "Shadow Softness Index", "05_shadow_softness_boxplot")


In [ ]:
def posthoc_heatmap(result, metric_label, filename):
    mat = result["posthoc_matrix"]
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(mat.astype(float), annot=True, fmt=".3f", cmap="coolwarm_r",
                vmin=0, vmax=1, center=0.05, ax=ax, cbar_kws={"label": "p-value"})
    ax.set_title(f"{metric_label} -- Post-hoc Comparison ({result['posthoc_method']})")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"{filename}.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(FIG_DIR, f"{filename}.pdf"), bbox_inches="tight")
    plt.show()

posthoc_heatmap(all_results["D"], "Fractal Dimension (D)", "06_posthoc_D")
posthoc_heatmap(all_results["Lac_mean"], "Mean Lacunarity", "07_posthoc_lacunarity")
posthoc_heatmap(all_results["SSI"], "Shadow Softness Index", "08_posthoc_SSI")


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.5))
corr = df[["D", "Lac_mean", "SSI"]].corr(method="spearman")
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation Between Metrics (Spearman)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "09_metric_correlation.png"), dpi=300, bbox_inches="tight")
plt.savefig(os.path.join(FIG_DIR, "09_metric_correlation.pdf"), bbox_inches="tight")
plt.show()


In [ ]:
zip_path = "/content/fractal_analysis_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_DIR):
        for f in files:
            full = os.path.join(root, f)
            arcname = os.path.relpath(full, OUTPUT_DIR)
            zf.write(full, arcname)

print(f"Zip ready: {zip_path}")
from google.colab import files
files.download(zip_path)
